<a href="https://colab.research.google.com/github/pedromilken/Disciplinas-Doutorado/blob/main/aula0_3_redes_neurais.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# PGC305 - Tópicos especiais em LLM e Deep Learning

## Definição dos dados

In [ ]:
import torch; import sklearn; from torch import nn

# 1. Carregar dados
iris = sklearn.datasets.load_iris()
X = iris.data        # 4 features: sépalas e pétalas
y = (iris.target == 1).astype(float)  # 1 se Versicolor, 0 caso contrário

# 2. Preparar dados para pytorch
X = torch.tensor(X, dtype=torch.float32)
y = torch.tensor(y, dtype=torch.float32).view(-1, 1)

## Definição do modelo e treinamento

In [ ]:
# 3. Definir modelo
import torch.nn.functional as F
class RedeNeural(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim):
        super(RedeNeural, self).__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x

In [ ]:
# Criar modelo
modelo = RedeNeural(4, 8, 1)  # 4 features → 1 saída (probabilidade de ser Versicolor)

import copy
modelo_clonado = copy.deepcopy(modelo)

learning_rate = 0.1

# Definir função de perda e algoritmo de otimização
funcao_perda = torch.nn.BCEWithLogitsLoss()  # combinação de sigmoid + BCE
optimizer = torch.optim.SGD(modelo.parameters(), lr=learning_rate)

## Execução do treinamento com optimizer SGD

In [ ]:
# Loop de treino
for epoch in range(1000):
    optimizer.zero_grad()           # Limpa gradientes
    outputs = modelo(X)             # Forward
    loss = funcao_perda(outputs, y) # Calcula perda
    loss.backward()                 # Calcula derivadas do gradiente
    optimizer.step()                # Aplica regra de alteração dos parâmetros

    if (epoch + 1) % 100 == 0:
        print(f"Época [{epoch+1}/100], Loss: {loss.item():.4f}")

# Treino com regra de gradiente descendente manual

In [ ]:
for epoch in range(1000):
    optimizer.zero_grad()           # Limpa gradientes
    outputs = modelo_clonado(X)     # Forward
    loss = funcao_perda(outputs, y) # Calcula perda
    loss.backward()                 # Calcula derivadas do gradiente

    with torch.no_grad():
        for param in modelo_clonado.parameters():
            param -= learning_rate * param.grad  # regra de atualização de pesos

    if (epoch + 1) % 100 == 0:
        print(f"Época [{epoch+1}/1000], Loss: {loss.item():.4f}")

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# 1. Carregar e Preparar Dados
iris = load_iris()
X = iris.data
# Criamos uma classificação binária: 1 para Versicolor, 0 para outras
y = (iris.target == 1).astype(float).reshape(-1, 1)

# Divisão Treino/Teste (80% treino, 20% teste)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Normalização (Essencial para Redes Neurais)
scaler = StandardScaler()
X_train = torch.tensor(scaler.fit_transform(X_train), dtype=torch.float32)
X_test = torch.tensor(scaler.transform(X_test), dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.float32)
y_test = torch.tensor(y_test, dtype=torch.float32)

# 2. Definição do Modelo
class ClassificadorIris(nn.Module):
    def __init__(self, input_dim=4, hidden_dim=16):
        super(ClassificadorIris, self).__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, 1) # Saída única (Logit)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        return self.fc2(x)

modelo = ClassificadorIris()

# 3. Configuração do Treino
# BCEWithLogitsLoss é mais estável que Sigmoid + BCELoss separadamente
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(modelo.parameters(), lr=0.01)

# 4. Loop de Treinamento
epochs = 200
for epoch in range(epochs):
    modelo.train()
    optimizer.zero_grad()

    # Forward pass
    outputs = modelo(X_train)
    loss = criterion(outputs, y_train)

    # Backward pass e otimização
    loss.backward()
    optimizer.step()

    if (epoch + 1) % 40 == 0:
        print(f"Época [{epoch+1}/{epochs}], Perda (Loss): {loss.item():.4f}")

# 5. Avaliação Final
modelo.eval() # Modo de avaliação (desativa dropout/batchnorm se houvesse)
with torch.no_grad():
    pred_logits = modelo(X_test)
    # Aplicamos sigmoid para converter logit em probabilidade (0 a 1)
    probs = torch.sigmoid(pred_logits)
    # Se prob > 0.5, classe 1, senão 0
    predicoes = (probs > 0.5).float()

    acuracia = (predicoes == y_test).sum() / y_test.size(0)
    print(f"\n--- Resultado Final ---")
    print(f"Acurácia no conjunto de teste: {acuracia.item() * 100:.2f}%")

# Multiclass

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# 1. Preparar Dados (Agora com 3 classes)
iris = load_iris()
X = iris.data
y = iris.target # Classes: 0 (Setosa), 1 (Versicolor), 2 (Virginica)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train = torch.tensor(scaler.fit_transform(X_train), dtype=torch.float32)
X_test = torch.tensor(scaler.transform(X_test), dtype=torch.float32)

# Importante: Para CrossEntropyLoss, o target 'y' deve ser LongTensor (inteiros)
y_train = torch.tensor(y_train, dtype=torch.long)
y_test = torch.tensor(y_test, dtype=torch.long)

# 2. Modelo Multiclasse
class ClassificadorIrisMulticlasse(nn.Module):
    def __init__(self, input_dim=4, hidden_dim=16, output_dim=3):
        super(ClassificadorIrisMulticlasse, self).__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, output_dim) # 3 saídas agora

    def forward(self, x):
        x = F.relu(self.fc1(x))
        return self.fc2(x) # Retorna Logits (sem ativação final aqui)

modelo = ClassificadorIrisMulticlasse()

# 3. Configuração (CrossEntropyLoss é a chave)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(modelo.parameters(), lr=0.01)

# 4. Treino
epochs = 200
for epoch in range(epochs):
    modelo.train()
    optimizer.zero_grad()

    outputs = modelo(X_train)
    loss = criterion(outputs, y_train)

    loss.backward()
    optimizer.step()

    if (epoch + 1) % 40 == 0:
        print(f"Época [{epoch+1}/{epochs}], Loss: {loss.item():.4f}")

# 5. Avaliação
modelo.eval()
with torch.no_grad():
    logits = modelo(X_test)
    # A classe predita é o índice com o maior valor (maior logit)
    _, predicoes = torch.max(logits, dim=1)

    acuracia = (predicoes == y_test).sum() / y_test.size(0)
    print(f"\n--- Resultado Final Multiclasse ---")
    print(f"Acurácia no teste: {acuracia.item() * 100:.2f}%")

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

# 1. Obter predições e targets em formato numpy
y_true = y_test.numpy()
y_pred = predicoes.numpy()

# 2. Gerar a matriz
cm = confusion_matrix(y_true, y_pred)

# 3. Visualizar com Seaborn
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=iris.target_names,
            yticklabels=iris.target_names)
plt.xlabel('Predito')
plt.ylabel('Real')
plt.title('Matriz de Confusão - Classificador Iris')
plt.show()

In [ ]:
# Exemplo de como ver as probabilidades reais
probabilidades = F.softmax(logits, dim=1)
print(probabilidades[0]) # Ex: tensor([0.0001, 0.9800, 0.0199]) -> 98% de chance de ser a classe 1